# Iris dataset — EDA, training and evaluation

This notebook performs a compact end-to-end pipeline for the Iris dataset: exploratory data analysis (EDA), model training (Random Forest pipeline), evaluation (confusion matrix) and a small prediction example. It uses `scikit-learn`, `pandas`, `seaborn`, and `matplotlib`.


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

%matplotlib inline

In [ ]:
# Load Iris dataset into a pandas DataFrame
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
df = pd.DataFrame(X, columns=feature_names)
df['species'] = pd.Categorical.from_codes(y, iris.target_names)

df.head()

## Basic statistics and visual EDA
We show summary statistics and a pairplot to visualise how features separate the species.


In [ ]:
# Summary statistics
df.describe().T

In [ ]:
# Pairplot (this may take a few seconds)
sns.pairplot(df, hue='species', corner=True, markers=['o','s','D'])
plt.suptitle('Pairplot of Iris features', y=1.02)
plt.show()

## Train a RandomForest pipeline
We split the data into train/test, build a `StandardScaler` + `RandomForestClassifier` pipeline, train it and save the model to `models/iris_model.joblib`.


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit
pipeline.fit(X_train, y_train)

# Evaluate on test set
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Save model
import os
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/iris_model.joblib')
print('Saved pipeline to models/iris_model.joblib')

## Confusion matrix on the whole dataset
We also show the confusion matrix evaluated on the full dataset (train+test) to get an overall view.


In [ ]:
# Confusion matrix on full dataset
full_pred = pipeline.predict(X)
cm = confusion_matrix(y, full_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (full dataset)')
plt.show()

## Quick prediction example
Try predicting one example by specifying measurements.


In [ ]:
# Single-sample prediction example
sample = [5.1, 3.5, 1.4, 0.2]  # sepal-length, sepal-width, petal-length, petal-width
pred = pipeline.predict([sample])[0]
proba = pipeline.predict_proba([sample])[0]

print(f'Sample: {sample}')
print(f'Predicted class: {iris.target_names[pred]} (index {pred})')
print(f'Predicted probabilities: {proba}')

---
**Next steps / ideas**:
- Add cross-validation and hyperparameter search (GridSearchCV).
- Add unit tests for the predict function.
- Package the model as a simple REST API using FastAPI or Flask.
